### 토스 NEXT ML CHALLENGE : 광고 클릭 예측(CTR) 모델 개발

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np


In [ ]:
train = pd.read_parquet('data/train.parquet', engine='pyarrow')

In [ ]:
train.head()

,gender,age_group,inventory_id,day_of_week,hour,seq,l_feat_1,l_feat_2,l_feat_3,l_feat_4,...,history_b_22,history_b_23,history_b_24,history_b_25,history_b_26,history_b_27,history_b_28,history_b_29,history_b_30,clicked
0,1.0,7.0,36,5,13,"9,18,269,516,57,97,527,74,317,311,269,479,57,7...",1.0,2.0,1.0,23.0,...,0.070092,0.070092,0.011682,0.004673,0.087226,0.049843,0.015576,0.040498,0.051401,0
1,1.0,7.0,2,5,08,"9,144,269,57,516,97,527,74,315,317,311,269,479...",2.0,2.0,3.0,17.0,...,0.072990,0.072990,0.012165,0.004866,0.045416,0.051904,0.016220,0.042172,0.026763,0
2,1.0,7.0,36,5,11,"269,516,57,97,165,527,74,77,317,269,75,450,15,...",1.0,2.0,1.0,7.0,...,0.057177,0.057177,0.009530,0.003812,0.035577,0.081318,0.012706,0.033036,0.062898,0
3,1.0,8.0,37,5,11,"269,57,516,21,214,269,561,214,269,561,247,516,...",2.0,2.0,2.0,7.0,...,0.100449,0.100449,0.016741,0.006697,0.062502,0.071430,0.022322,0.058037,0.073659,0
4,2.0,7.0,37,5,07,"144,269,57,516,35,479,57,516,527,74,77,318,193...",2.0,2.0,3.0,24.0,...,0.064512,0.064512,0.010752,0.004301,0.040141,0.045875,0.014336,0.037274,0.023654,0


In [9]:
train.describe()

,l_feat_1,l_feat_2,l_feat_3,l_feat_4,l_feat_5,l_feat_6,l_feat_7,l_feat_8,l_feat_9,l_feat_10,...,history_b_22,history_b_23,history_b_24,history_b_25,history_b_26,history_b_27,history_b_28,history_b_29,history_b_30,clicked
count,1.070418e+07,1.068697e+07,1.070418e+07,1.070418e+07,1.070418e+07,1.070418e+07,1.070418e+07,1.068697e+07,1.070418e+07,1.070418e+07,...,1.068697e+07,1.068697e+07,1.068697e+07,1.068697e+07,1.068697e+07,1.068697e+07,1.068697e+07,1.068697e+07,1.068697e+07,1.070418e+07
mean,1.858145e+00,1.831941e+00,2.357217e+00,9.543341e+00,3.938106e+02,3.147881e+02,1.479096e+02,1.994025e+00,2.128026e+02,1.239446e+02,...,4.416593e-01,4.396585e-01,7.326672e-02,2.928779e-02,2.777950e-01,3.167809e-01,9.889161e-02,2.540546e-01,2.173098e-01,1.907470e-02
std,3.489011e-01,3.739182e-01,7.164651e-01,5.946347e+00,2.899930e+02,2.313294e+02,7.192693e+01,7.706346e-02,1.251033e+02,6.209058e+01,...,1.987997e+00,1.986785e+00,3.311270e-01,1.324596e-01,1.237975e+00,1.414600e+00,4.418534e-01,1.147925e+00,7.643794e-01,1.367876e-01
min,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.000000e+00,2.000000e+00,2.000000e+00,7.000000e+00,1.050000e+02,1.090000e+02,9.400000e+01,2.000000e+00,1.010000e+02,8.300000e+01,...,5.310000e-02,5.250600e-02,8.751000e-03,3.484200e-03,3.454640e-02,3.886080e-02,1.236800e-02,3.039400e-02,4.411770e-02,0.000000e+00
50%,2.000000e+00,2.000000e+00,2.000000e+00,7.000000e+00,3.760000e+02,2.780000e+02,1.460000e+02,2.000000e+00,2.240000e+02,1.250000e+02,...,9.345600e-02,9.240300e-02,1.540050e-02,6.135000e-03,6.100080e-02,6.874240e-02,2.164600e-02,5.344040e-02,7.845420e-02,0.000000e+00
75%,2.000000e+00,2.000000e+00,3.000000e+00,1.200000e+01,6.520000e+02,4.710000e+02,2.120000e+02,2.000000e+00,3.090000e+02,1.790000e+02,...,2.088180e-01,2.064240e-01,3.440400e-02,1.372980e-02,1.342936e-01,1.527424e-01,4.750600e-02,1.192672e-01,1.506846e-01,0.000000e+00
max,2.000000e+00,2.000000e+00,3.000000e+00,2.600000e+01,1.079000e+03,9.030000e+02,3.130000e+02,2.000000e+00,4.760000e+02,2.620000e+02,...,4.500000e+01,4.500000e+01,7.500000e+00,3.000000e+00,2.800000e+01,3.200000e+01,1.000000e+01,2.600000e+01,3.300000e+01,1.000000e+00


In [ ]:
train.select_dtypes(include='object').nunique()